# Prompt evaluation

Requirements:
- Use preferrably Python `3.12.13` venvs.
    - Using older version might not stream events chunk by chunk, instead it might wait for the **entire** response before printing it.
- Add a `.env` file in the same folder as this notebook with a fields:
    - ANTHROPIC_BASE_URL = `<url_here>`
    - ANTHROPIC_AUTH_TOKEN = "`<jwt_token_here>`"

## ATENTION

- Prefer "claude-4-5-haiku" for this notebook.

## 0. Setup

### 0.0 Libraries

In [1]:
from dotenv import load_dotenv
from anthropic import Anthropic

### 0.1. Env and Client

In [2]:
load_dotenv()
client = Anthropic()
model = "bedrock/anthropic.claude-4-5-haiku"

### 0.2. Helper functions

In [3]:
def add_user_message(
    messages, 
    text,
):
    user_message = {
        "role": "user", 
        "content": text,
    }
    
    messages.append(user_message)

In [4]:
def add_assistant_message(
    messages, 
    text,
):
    assistant_message = {
        "role": "assistant", 
        "content": text,
    }

    messages.append(assistant_message)

In [5]:
def chat(
    messages, 
    system=None, 
    temperature=1.0, 
    stop_sequences=[],
):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system
    
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

## 1. Prompt eval dataset

### 1.0. Libraries

In [6]:
import json

### 1.1. Generate dataset
- Task object
    - "task": String

In [7]:
def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []

    add_user_message(
        messages, 
        prompt,
    )

    prefill = "```json"
    add_assistant_message(
        messages,
        prefill,
    )

    stop_sequences = ["```"]
    text = chat(
        messages,
        stop_sequences=stop_sequences,
    )

    return json.loads(text)

### 1.2. Generate dataset and save to file

In [8]:
# dataset = generate_dataset()

In [9]:
# with open('dataset.json', 'w') as f:
#     json.dump(
#         dataset, 
#         f, 
#         indent=2,
#     )

# print("Dataset saved to dataset.json")

## 2. Running the eval

### 2.1. Helpers

In [10]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [11]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [12]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

### 2.2. Read dataset

In [13]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

### 2.3. Run eval

In [14]:
results = run_eval(dataset)

In [15]:
print(json.dumps(results, indent=1))

[
 {
  "output": "# Parse AWS S3 Bucket Name from URI\n\nHere's a solution that extracts the bucket name from an S3 URI:\n\n```python\ndef parse_s3_bucket_name(s3_uri):\n    \"\"\"\n    Extract the bucket name from an S3 URI.\n    \n    Args:\n        s3_uri (str): Full S3 URI (e.g., 's3://my-bucket/path/to/file.txt')\n    \n    Returns:\n        str: The bucket name\n    \n    Raises:\n        ValueError: If the URI format is invalid\n    \"\"\"\n    if not s3_uri.startswith('s3://'):\n        raise ValueError(\"Invalid S3 URI: must start with 's3://'\")\n    \n    # Remove 's3://' prefix\n    uri_without_prefix = s3_uri[5:]\n    \n    # Split by '/' and get the first part (bucket name)\n    bucket_name = uri_without_prefix.split('/')[0]\n    \n    if not bucket_name:\n        raise ValueError(\"Invalid S3 URI: bucket name is empty\")\n    \n    return bucket_name\n\n\n# Test cases\nif __name__ == \"__main__\":\n    # Basic example\n    print(parse_s3_bucket_name('s3://my-bucket/path/

## 3. Model based grader

### 3.0. Libraries

In [16]:
from statistics import mean

### 3.1. Helpers
- new grade_by_model() function

In [17]:
def grade_by_model(
    test_case,
    output,
): 
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []

    prefill_sequence = "```json"
    stop_sequence = "```"

    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, prefill_sequence)

    eval_text = chat(
        messages,
        temperature=0,
        stop_sequences=[stop_sequence],
    )

    return json.loads(eval_text)

### 3.2. Overrides
- run_test_case(): now uses model grader instead of giving 10/10s
- run_eval(): now averages dataset's grades

In [18]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # Grader
    model_grade = grade_by_model(
        test_case,
        output,
    )

    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    strengths = model_grade["strengths"]
    weaknesses = model_grade["weaknesses"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
        "strengths": strengths,
        "weaknesses": weaknesses,
    }

In [19]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    # Average grader score
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score:.2f}")

    return results

### 3.3. Read dataset

In [20]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

### 3.4. Run eval

In [21]:
results = run_eval(dataset)

Average score: 7.67


In [22]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket Name Parser\n\nHere's a solution to extract the bucket name from an S3 URI:\n\n```python\ndef parse_s3_bucket_name(s3_uri):\n    \"\"\"\n    Extract the bucket name from an S3 URI.\n    \n    Args:\n        s3_uri (str): Full S3 URI (e.g., 's3://my-bucket/path/to/file.txt')\n    \n    Returns:\n        str: The bucket name, or None if invalid format\n    \n    Raises:\n        ValueError: If the URI is not a valid S3 path\n    \"\"\"\n    if not s3_uri.startswith('s3://'):\n        raise ValueError(f\"Invalid S3 URI format: {s3_uri}\")\n    \n    # Remove 's3://' prefix\n    uri_without_protocol = s3_uri[5:]\n    \n    # Extract bucket name (everything before the first '/')\n    bucket_name = uri_without_protocol.split('/')[0]\n    \n    if not bucket_name:\n        raise ValueError(\"Bucket name is empty\")\n    \n    return bucket_name\n\n\n# Test cases\nif __name__ == \"__main__\":\n    test_cases = [\n        \"s3://my-bucket/path/to/file.txt\",

## 4. Code grader

### 4.0. Libraries

In [23]:
import json

# Model grader
from statistics import mean

# Code grader
import re
import ast

### 4.1 New dataset with "format" field

In [24]:
def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []

    add_user_message(
        messages, 
        prompt,
    )

    prefill = "```json"
    add_assistant_message(
        messages,
        prefill,
    )

    stop_sequences = ["```"]
    text = chat(
        messages,
        stop_sequences=stop_sequences,
    )

    return json.loads(text)

In [25]:
# dataset = generate_dataset()

In [26]:
# with open("dataset.json", "w") as f:
#     json.dump(dataset, f, indent=2)

### 4.2. Code graders

In [27]:
def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

In [28]:
def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

In [29]:
def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

In [30]:
def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

### 4.3 Helper overrides
- run_prompt(): now prompts for a strictly formatted response.
- run_test_case(): now grades both by model and code.

In [31]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""
    
    messages = []

    add_user_message(messages, prompt)

    prefill = "```code"
    add_assistant_message(messages, prefill)

    stop_sequence = "```"
    output = chat(messages, stop_sequences=[stop_sequence])
    
    return output

In [32]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # Model grader
    model_grade = grade_by_model(
        test_case,
        output,
    )

    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    strengths = model_grade["strengths"]
    weaknesses = model_grade["weaknesses"]

    # Code grader
    syntax_score = grade_syntax(
        output, 
        test_case,
    )

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
        "strengths": strengths,
        "weaknesses": weaknesses,
    }

### 4.4 Run complete eval

In [33]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

In [34]:
results = run_eval(dataset)

Average score: 8.33


In [35]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport re\n\ndef extract_s3_bucket_name(s3_uri):\n    match = re.match(r's3://([^/]+)', s3_uri)\n    return match.group(1) if match else None\n",
    "test_case": {
      "task": "Parse an AWS S3 bucket name from a full S3 URI (e.g., 's3://my-bucket/path/to/file.txt') and extract just the bucket name",
      "format": "regex"
    },
    "score": 8.5,
    "reasoning": "The solution correctly solves the core task with a well-crafted regex pattern and appropriate null handling. However, it lacks defensive programming practices (input validation, type hints) and doesn't account for alternative S3 URI schemes used in production environments. The code would benefit from basic documentation and more robust error handling.",
    "strengths": [
      "Correct regex pattern that accurately extracts bucket names from S3 URIs using s3://([^/]+) to match everything between 's3://' and the first '/'",
      "Handles edge cases gracefully by returning None when the URI doesn't 

## 5. Exercise - improve evaluation with more context

### 5.0 Libraries

In [36]:
import json

# Model grader
from statistics import mean

# Code grader
import re
import ast

### 5.1. Add "solution_criteria" field to dataset generation function
- task
- format
- solution_criteria (NEW!)

In [37]:
def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Description of what a correct solution should look like",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []

    add_user_message(
        messages, 
        prompt,
    )

    prefill = "```json"
    add_assistant_message(
        messages,
        prefill,
    )

    stop_sequences = ["```"]
    text = chat(
        messages,
        stop_sequences=stop_sequences,
    )

    return json.loads(text)

In [38]:
# dataset = generate_dataset()
# print(dataset)

In [39]:
# with open('dataset_exercise.json', 'w') as f:
#     json.dump(
#         dataset, 
#         f, 
#         indent=2,
#     )

# print("Dataset saved to dataset_exercise.json")

### 5.2. Include "solution criteria" in model grader.

In [40]:
def grade_by_model(
    test_case,
    output,
): 
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria to use for evaluation:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []

    prefill_sequence = "```json"
    stop_sequence = "```"

    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, prefill_sequence)

    eval_text = chat(
        messages,
        temperature=0,
        stop_sequences=[stop_sequence],
    )

    return json.loads(eval_text)

### 5. Evaluation helpers
- **Run prompt**: model response for a test case.
- **Run test** case: evaluate model response for a single test case, using both model and code graders.
- **Run eval**: run each test case from the dataset.
- **code graders**

In [41]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""
    
    messages = []

    add_user_message(messages, prompt)

    prefill = "```code"
    add_assistant_message(messages, prefill)

    stop_sequence = "```"
    output = chat(messages, stop_sequences=[stop_sequence])
    
    return output

In [42]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # Model grader
    model_grade = grade_by_model(
        test_case,
        output,
    )

    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    strengths = model_grade["strengths"]
    weaknesses = model_grade["weaknesses"]

    # Code grader
    syntax_score = grade_syntax(
        output, 
        test_case,
    )

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "solution_criteria": test_case["solution_criteria"],
        "score": score,
        "reasoning": reasoning,
        "strengths": strengths,
        "weaknesses": weaknesses,
    }

In [43]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    # Average grader score
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score:.2f}")

    return results

In [44]:
def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

In [45]:
def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

In [46]:
def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

In [47]:
def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

### 5. Run exercise evaluation - solution criteria

In [48]:
with open("dataset_exercise.json", "r") as f:
    dataset = json.load(f)

In [49]:
results = run_eval(dataset)

Average score: 8.50


In [50]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport re\nimport json\n\ndef extract_s3_bucket_names(template_str):\n    \"\"\"\n    Extract AWS S3 bucket names from CloudFormation template resource names\n    that follow the pattern 'MyBucket[BucketName]Stack'\n    \"\"\"\n    pattern = r'MyBucket(\\w+)Stack'\n    matches = re.findall(pattern, template_str)\n    return matches\n\n# Example usage with CloudFormation template\nif __name__ == \"__main__\":\n    # Test with sample CloudFormation template\n    sample_template = \"\"\"\n    {\n        \"Resources\": {\n            \"MyBucketDataStack\": {...},\n            \"MyBucketLogsStack\": {...},\n            \"MyBucketBackupStack\": {...},\n            \"OtherResourceName\": {...}\n        }\n    }\n    \"\"\"\n    \n    bucket_names = extract_s3_bucket_names(sample_template)\n    print(json.dumps(bucket_names, indent=2))\n",
    "test_case": {
      "task": "Extract all AWS S3 bucket names from CloudFormation template resource names that follow the pattern